# Ahmed Twin — DPO Training (v8: self-diagnosing, clean dev/test split)

DPO (Direct Preference Optimization) on top of the NileChat-3B SFT model, using hard-negative preference pairs (`chosen` = Ahmed's real reply, `rejected` = a real reply he sent in a *different* conversation).

**Design goal of v7:** stop fixing things one run at a time. Every run now measures itself against a clean baseline, keeps its own artifacts, and ends with an automatic diagnosis that says which knob to turn next.

### v8 changes (from the v7 run)

| v7 result | v8 change |
|---|---|
| Held-out accuracy +2.7 points (64.7% → 67.3%), but on 150 pairs the noise is about ±4 points, so the diagnosis said "OK" for a result that wasn't yet proven | The diagnosis now runs a paired significance test (which pairs flipped from wrong to right, and vice versa) and says whether the gain is real or within noise |
| Only 150 of the 200 eval pairs were scored | All eval pairs are scored by default (more pairs, less noise) |
| `logps/chosen` stayed stable (-2.2%, vs -8.7% in v6), so the anchor works, and there's room to let the model move further | Next experiment: `beta` 0.2 → 0.1, as a new run (`dpo_v8_beta01`) so v7 is kept for comparison |
| The merge cell failed (Unsloth requires loading the adapter folder directly, not through `PeftModel`) | Merge cell fixed, and it frees GPU memory first |
| Comparing runs meant opening several JSON files by hand | New final cell: a comparison table of every run in `runs/` |
| The same 200 pairs were used both to **pick** the best checkpoint (early stopping) and to **report** held-out accuracy, which biases the result upward | Separate **dev** set (picks the checkpoint) and **test** set (reports the result). The test set is never used for any decision during training |
| v7 and v8 would have been compared on different pairs (150 vs 200), each against its own baseline | New head-to-head cell: scores other runs' adapters (e.g. v7) on **this** run's test pairs and tests the difference pair by pair |
| "Best checkpoint restored" was assumed, not checked | The restored weights are compared against the checkpoint file on disk, and reloaded if they don't match |
| Diagnosis measured drift from step 25, not from the start | An evaluation at step 0 (before any training) is the starting point for the diagnosis |

### What changed from v6, and why (each item is backed by evidence from the v6 run)

| Problem seen in v6 | v7 fix |
|---|---|
| `logps/chosen` fell from -49.9 to -54.2 during training: DPO made Ahmed's *real* replies **less** likely and won only by pushing `rejected` down faster (known as *likelihood displacement*) | Adds an SFT anchor on `chosen` (`rpo_alpha`), which keeps real replies likely while learning the preference |
| `beta=0.5` + 3 fixed epochs barely changed generations (2 of 5 test prompts identical) | `beta` lowered to `0.2`. This is safe now **because** of the anchor above |
| Validation pairs came from `train.jsonl`, which the SFT model already memorized, so the eval could not measure generalization | Dev and test sets built from `validation.jsonl` (never seen by SFT), plus overlap checks against training |
| DPO's built-in "reward accuracy" is 0 by definition at step 0 (policy = reference), so there was no real baseline | New **held-out preference accuracy**: does the model itself assign higher likelihood to the real reply than to the wrong one? Measured before **and** after training |
| Fixed 3 epochs; the best checkpoint could be deleted by cleanup | `load_best_model_at_end` + early stopping. Checkpoint deletion is now a manual, opt-in step |
| Every run overwrote the previous adapter | One folder per run (`RUN_NAME`) with a `run_summary.json` of config, metrics, and diagnosis |
| "1500 / 1500 rows could not be parsed" false alarm | Format-aware prompt check: already-templated rows are recognized, and only real failures stop the run |
| Possible double EOS (newer TRL appends EOS itself) | EOS handling is verified on the actual tokenized data instead of being assumed |
| Tokenizer "incorrect regex pattern" warning on load | Tokenizer is compared against the original `UBC-NLP/NileChat-3B` tokenizer on real data |
| `temperature` was silently ignored (no `do_sample`) | Before/after generation is explicitly greedy, so differences come from the weights, not sampling luck |

### Before running
1. Training pairs: `dpo_dataset_hard.jsonl` (already built from `train.jsonl`).
2. **Dev pairs** (early stopping): `dpo_eval_hard.jsonl`, 200 pairs from `validation.jsonl`, already built. Keep it as is: v7 used exactly this file, so v7 and v8 pick checkpoints the same way.
3. **Test pairs** (final result only): build once from `validation.jsonl` with more samples. Pairs that overlap with dev or train are removed automatically, leaving about 300 fresh ones:
   `python scripts/build_hard_negative_dpo_dataset.py --train-path validation.jsonl --output-path dpo_test_hard.jsonl --num-samples 500`
4. Upload all three files to `DPO/` on Drive. Set `RUN_NAME` to something new for every experiment.
5. **Before committing this notebook to git: Edit → Clear all outputs.** A previous run's output captured part of a token.

---
## Part A — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q unsloth trl datasets

### A1. Imports, GPU check, seeds

`unsloth` must be imported before `trl`, `transformers`, and `peft`, or its optimizations are skipped (v6 printed a warning about this). `UNSLOTH_RETURN_LOGITS=1` makes sure the model returns real logits; the held-out preference evaluation below depends on them.

In [ ]:
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

import unsloth  # must come before trl / transformers / peft
from unsloth import FastLanguageModel, PatchDPOTrainer, is_bfloat16_supported
PatchDPOTrainer()

import json, glob, shutil, random
import numpy as np
import torch
from transformers import set_seed

assert torch.cuda.is_available(), "No GPU detected -- Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

SEED = 0
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

### A2. Run configuration — the only cell you should need to edit

Every knob lives here, so a run is fully described by this cell and recorded in `run_summary.json`.

**Decision guide.** Part F prints an automatic diagnosis; this is how to act on it:

| Diagnosis | Knob to turn |
|---|---|
| Held-out preference accuracy barely moved | Lower `beta` (0.2 → 0.1), or build more pairs |
| `logps/chosen` still falling noticeably | Raise `rpo_alpha` (0.5 → 1.0) |
| Accuracy ≥98% early in training | Dataset too easy: tighten the builder (`--max-similarity 0.4`) |
| Best checkpoint is early, eval loss rises after | Fine: early stopping handled it. If it stops at the first eval, lower `learning_rate` |
| Held-out accuracy went **down** | Overfitting or a data mismatch: lower `learning_rate`, check the train/eval overlap report |

In [ ]:
# ======================= RUN CONFIG =======================
RUN_NAME = "dpo_v8_beta01"         # change for EVERY new experiment (v7 = "dpo_v7_rpo", kept for comparison)
OVERWRITE_EXISTING_RUN = False     # refuses to overwrite a finished run unless True

DRIVE_ROOT = "/content/drive/MyDrive"
BASE_CHECKPOINT_PATH = f"{DRIVE_ROOT}/ahmed_twin/model_merged_nilechat"   # SFT model (checkpoint-1000, merged)
TRAIN_JSONL_PATH = f"{DRIVE_ROOT}/DPO/dpo_dataset_hard.jsonl"            # built from train.jsonl
DEV_JSONL_PATH = f"{DRIVE_ROOT}/DPO/dpo_eval_hard.jsonl"                 # early stopping only (validation.jsonl, 200 pairs)
TEST_JSONL_PATH = f"{DRIVE_ROOT}/DPO/dpo_test_hard.jsonl"                # final result only (validation.jsonl, never used in training decisions)
ORIGINAL_TOKENIZER = "UBC-NLP/NileChat-3B"                               # only for the tokenizer sanity check

RUN_DIR = f"{DRIVE_ROOT}/DPO/runs/{RUN_NAME}"
OUTPUT_DIR = f"{RUN_DIR}/checkpoints"
ADAPTER_DIR = f"{RUN_DIR}/adapter"
MERGED_MODEL_DIR = f"{RUN_DIR}/merged"
SUMMARY_PATH = f"{RUN_DIR}/run_summary.json"

MAX_SEQ_LENGTH = 2048     # model context used during SFT
MAX_LENGTH = 1024         # prompt + completion budget for DPO
MAX_PROMPT_LENGTH = 768   # leaves room for the reply; the truncation report below checks this

HP = dict(
    beta=0.1,                     # v6: 0.5 (too little movement), v7: 0.2 (+2.7 pts, chosen stayed likely)
    rpo_alpha=0.5,                # SFT anchor on `chosen`; fixes likelihood displacement. 0 = plain DPO
    learning_rate=1e-5,
    num_train_epochs=3,           # a ceiling only -- early stopping usually ends sooner
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    warmup_steps=10,
    eval_steps=25,                # evaluate + checkpoint every N steps (must match for best-model tracking)
    early_stopping_patience=3,    # stop after N evals without eval-loss improvement
)

COMPARE_WITH_RUNS = ["dpo_v7_rpo"] # finished runs to score head-to-head on THIS run's test pairs (G3)
N_HELDOUT_GENERATIONS = 5         # real held-out conversations to generate replies for

SYSTEM_MESSAGE = (
    "Continue the conversation as Ahmed. Match the language, tone, and "
    "context of the conversation. Do not claim to know facts that are not "
    "supported by the dialogue."
)
HANDWRITTEN_TEST_MESSAGES = [
    "ممكن تساعدني في حاجة مهمة النهاردة؟ محتاج رأيك بجد",
    "معلش تأخرت في الرد كنت مشغول شوية",
    "ايه رأيك في اللي حصل النهاردة في الشغل؟",
    "مش عارف اعمل ايه في القرار ده ممكن تنصحني",
    "بجد شكرا انك كنت جنبي في الموضوع ده",
]

### A3. Run-folder guard and disk check

Detects whether this `RUN_NAME` already finished. If it did, **training** is blocked (in E3) so the finished run can't be overwritten, but head-to-head comparison (G3), merging (G5), and the run table (G6) still work. That lets you open a fresh session, point `RUN_NAME` at the winning run, and merge it. The cell also fails up front if Drive is too full to hold checkpoints; in earlier runs, a full Drive caused silent save failures.

In [ ]:
RUN_ALREADY_FINISHED = os.path.exists(os.path.join(ADAPTER_DIR, "adapter_model.safetensors"))
if RUN_ALREADY_FINISHED:
    print(f"NOTE: run '{RUN_NAME}' already has a saved adapter at {ADAPTER_DIR}.")
    print("Training (E3) is blocked for this run name; comparing (G3, G6) and merging (G5) work as normal.")
    print("To train a new experiment, set a new RUN_NAME. To retrain this one on purpose, set OVERWRITE_EXISTING_RUN = True.")
os.makedirs(RUN_DIR, exist_ok=True)

free_gb = shutil.disk_usage(DRIVE_ROOT).free / 1e9
print(f"Free Drive space: {free_gb:.2f} GB")
assert free_gb > 3, (
    f"Only {free_gb:.2f} GB free. Checkpoints (~0.4 GB each, up to ~3 kept) plus the adapter "
    "need roughly 2-3 GB. Free space first."
)
print("Run folder:", RUN_DIR)

---
## Part B — Model and tokenizer

### B1. Hugging Face login

Only needed for the tokenizer sanity check (the SFT model itself loads from Drive). The token is entered at runtime and never stored in the notebook.

In [ ]:
from huggingface_hub import login
from getpass import getpass

login(token=getpass("Hugging Face token: "))

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_CHECKPOINT_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
)
assert getattr(tokenizer, "chat_template", None), "Tokenizer has no chat template -- cannot build matching prompts."


def resolve_eos_id(tok, preferred="<|im_end|>"):
    token_id = tok.convert_tokens_to_ids(preferred)
    if token_id is None or token_id == tok.unk_token_id:
        return tok.eos_token_id
    return token_id


EOS_ID = resolve_eos_id(tokenizer)
print("Loaded:", BASE_CHECKPOINT_PATH)
print("eos_token:", repr(tokenizer.eos_token), "| chat-turn end id used for generation:", EOS_ID)

### B2. Attach LoRA adapters and fingerprint the starting weights

This uses the same LoRA setup as SFT. A freshly attached adapter starts with zero effect on the output, so every "before" measurement taken from here until training is a measurement of the plain SFT model.

The fingerprint (sum of absolute LoRA weights) catches a silent no-op run. An earlier run "completed" while changing almost nothing.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"],
    random_state=SEED,
    use_gradient_checkpointing="unsloth",
)


def lora_fp(m):
    return sum(p.detach().float().abs().sum().item() for n, p in m.named_parameters() if "lora_" in n)


def adapter_file_fp(adapter_file):
    # Fingerprint of an adapter file on disk -- same quantity as lora_fp, so the
    # two can be compared to prove which weights are actually loaded in the model.
    from safetensors.torch import load_file
    state = load_file(adapter_file)
    return sum(t.float().abs().sum().item() for k, t in state.items() if "lora_" in k), state


def load_adapter_weights(adapter_path):
    # Loads LoRA weights from an adapter folder or .safetensors file into `model`,
    # then PROVES it worked by matching fingerprints (raises if it didn't).
    from peft import set_peft_model_state_dict
    path = adapter_path if adapter_path.endswith(".safetensors") else os.path.join(adapter_path, "adapter_model.safetensors")
    expected, state = adapter_file_fp(path)
    set_peft_model_state_dict(model, state)
    loaded = lora_fp(model)
    assert abs(loaded - expected) / expected < 1e-3, (
        f"Adapter load failed: model fingerprint {loaded:.2f} != file fingerprint {expected:.2f} ({path})"
    )
    return loaded


fp_before = lora_fp(model)
print("LoRA fingerprint BEFORE training:", round(fp_before, 2))

---
## Part C — Data: load, validate, normalize

### C1. Load training, dev, and test pairs

- **train**: what DPO learns from (from `train.jsonl`)
- **dev**: what the trainer evaluates during training to pick the best checkpoint and stop early (from `validation.jsonl`)
- **test**: only used to measure the final result, before and after training (from `validation.jsonl`, overlap with dev/train removed)

Keeping dev and test separate matters: if the same pairs choose the checkpoint *and* report the result, the result comes out optimistic.

Fallbacks: if the test file is missing, the dev set is split in half (noisier, with a warning). If the dev file is also missing, both come from the training file and the run is marked **not clean**.

In [ ]:
from datasets import load_dataset

train_ds = load_dataset("json", data_files=TRAIN_JSONL_PATH, split="train")
CLEAN_EVAL = os.path.exists(DEV_JSONL_PATH)
SEPARATE_TEST = os.path.exists(TEST_JSONL_PATH)

if CLEAN_EVAL:
    dev_ds = load_dataset("json", data_files=DEV_JSONL_PATH, split="train")
    if SEPARATE_TEST:
        test_ds = load_dataset("json", data_files=TEST_JSONL_PATH, split="train")
    else:
        print("WARNING: no test file at", TEST_JSONL_PATH)
        print("Splitting the dev set in half instead -> fewer test pairs (more noise), and the")
        print("head-to-head with other runs (G3) is biased: they used these pairs for early stopping.")
        halves = dev_ds.train_test_split(test_size=0.5, seed=SEED)
        dev_ds, test_ds = halves["train"], halves["test"]
else:
    print("WARNING: no dev file at", DEV_JSONL_PATH, "-- falling back to splits of the TRAINING file.")
    print("These prompts were seen during SFT, so results will look better than real generalization.")
    split = train_ds.train_test_split(test_size=0.2, seed=SEED)
    train_ds = split["train"]
    halves = split["test"].train_test_split(test_size=0.5, seed=SEED)
    dev_ds, test_ds = halves["train"], halves["test"]

print(f"Train: {len(train_ds)} | Dev: {len(dev_ds)} | Test: {len(test_ds)} (before validation/overlap checks)")

### C2. Validate pairs and check for train/eval overlap

Empty fields stop the run. Identical chosen/rejected pairs carry no learning signal and are dropped. Dev pairs whose prompt appears in training are removed, and test pairs whose prompt appears in training **or** dev are removed, so the test set really is untouched. (The test file is built with the same seed as the dev file, so its first ~200 pairs repeat dev; that's expected, and they're dropped here.)

In [ ]:
def validate_pairs(ds, name):
    required = {"prompt", "chosen", "rejected"}
    missing = required - set(ds.column_names)
    assert not missing, f"{name}: missing columns {missing}"

    empty = sum(1 for ex in ds if not ex["chosen"].strip() or not ex["rejected"].strip() or not ex["prompt"].strip())
    if empty:
        raise ValueError(f"{name}: {empty} rows with an empty prompt/chosen/rejected -- fix the dataset first.")

    before = len(ds)
    ds = ds.filter(lambda ex: ex["chosen"].strip() != ex["rejected"].strip())
    if before - len(ds):
        print(f"{name}: dropped {before - len(ds)} identical chosen/rejected pairs")
    print(f"{name}: {len(ds)} valid pairs")
    return ds


train_ds = validate_pairs(train_ds, "train")
dev_ds = validate_pairs(dev_ds, "dev")
test_ds = validate_pairs(test_ds, "test")

train_prompts = set(train_ds["prompt"])
dev_before = len(dev_ds)
dev_ds = dev_ds.filter(lambda ex: ex["prompt"] not in train_prompts)

seen_prompts = train_prompts | set(dev_ds["prompt"])
test_before = len(test_ds)
test_ds = test_ds.filter(lambda ex: ex["prompt"] not in seen_prompts)

print(f"Dev:  removed {dev_before - len(dev_ds)} overlapping with train -> {len(dev_ds)} pairs")
print(f"Test: removed {test_before - len(test_ds)} overlapping with train/dev -> {len(test_ds)} pairs")
assert len(dev_ds) >= 50, "Dev set too small after overlap removal."
assert len(test_ds) >= 50, "Test set too small after overlap removal -- build dpo_test_hard.jsonl with more --num-samples."
if len(test_ds) < 200:
    print(f"NOTE: {len(test_ds)} test pairs is on the small side; differences of a few points will read as noise.")

### C3. Make sure every prompt is in the real chat-template format

The hard-negative builder already writes prompts in the tokenizer's chat template (`<|im_start|>...`). Older datasets used a plain `role: content` format, which caused a train/eval format mismatch earlier in the project.

This step recognizes both formats. Templated rows pass through, and plain rows are converted. Only rows that fit neither format stop the run. That replaces v6's misleading "1500 / 1500 could not be parsed" message.

In [ ]:
TEMPLATE_MARKER = "<|im_start|>"
GENERATION_SUFFIX = "<|im_start|>assistant\n"


def plain_to_template(prompt_text):
    lines = prompt_text.strip().split("\n")
    if lines and lines[-1].strip() == "assistant:":
        lines = lines[:-1]
    messages = []
    for line in lines:
        role, sep, content = line.partition(": ")
        if sep and role.strip() in ("system", "user", "assistant"):
            messages.append({"role": role.strip(), "content": content.strip()})
        elif messages:
            messages[-1]["content"] += "\n" + line   # continuation of a multi-line message
    if not messages:
        return None
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def normalize_prompts(ds, name):
    counts = {"already_templated": 0, "converted": 0, "unparseable": 0, "bad_ending": 0}

    def fix(ex):
        p = ex["prompt"]
        if TEMPLATE_MARKER in p:
            counts["already_templated"] += 1
        else:
            converted = plain_to_template(p)
            if converted is None:
                counts["unparseable"] += 1
                return ex
            counts["converted"] += 1
            ex["prompt"] = converted
        if not ex["prompt"].endswith(GENERATION_SUFFIX):
            counts["bad_ending"] += 1
        return ex

    ds = ds.map(fix, load_from_cache_file=False)
    print(f"{name}: {counts}")
    assert counts["unparseable"] == 0, f"{name}: {counts['unparseable']} prompts in an unknown format -- inspect before training."
    assert counts["bad_ending"] == 0, f"{name}: {counts['bad_ending']} prompts don't end with the assistant generation prompt."
    return ds


train_ds = normalize_prompts(train_ds, "train")
dev_ds = normalize_prompts(dev_ds, "dev")
test_ds = normalize_prompts(test_ds, "test")

# Re-check overlap now that every prompt is in the same format: a converted legacy prompt
# could only become identical to a train/dev prompt after conversion.
train_prompts = set(train_ds["prompt"])
dev_ds = dev_ds.filter(lambda ex: ex["prompt"] not in train_prompts)
seen_prompts = train_prompts | set(dev_ds["prompt"])
n_before = len(test_ds)
test_ds = test_ds.filter(lambda ex: ex["prompt"] not in seen_prompts)
if len(test_ds) != n_before:
    print(f"Removed {n_before - len(test_ds)} more test pairs that only matched after format conversion.")
print(f"Final sizes -> train: {len(train_ds)} | dev: {len(dev_ds)} | test: {len(test_ds)}")

### C4. Tokenizer sanity check

On load, the tokenizer warned about an "incorrect regex pattern". This cell checks whether that warning matters for *this* data by comparing tokenization against the original `UBC-NLP/NileChat-3B` tokenizer on real replies. If they match, the warning is harmless here. If they don't, the saved tokenizer is splitting Arabic differently than the model was pretrained on, and that should be fixed before training.

In [ ]:
from transformers import AutoTokenizer

reference_tokenizer = AutoTokenizer.from_pretrained(ORIGINAL_TOKENIZER)
sample_texts = list(train_ds["chosen"][:100]) + list(train_ds["rejected"][:100])

mismatches = [
    t for t in sample_texts
    if reference_tokenizer(t, add_special_tokens=False).input_ids != tokenizer(t, add_special_tokens=False).input_ids
]
print(f"Tokenization mismatches vs. original tokenizer: {len(mismatches)} / {len(sample_texts)}")
if mismatches:
    print("Example mismatch:", mismatches[0][:120])
    raise RuntimeError(
        "The saved tokenizer tokenizes real data differently from the original. Re-save the SFT model's "
        "tokenizer from the original (AutoTokenizer.from_pretrained(ORIGINAL_TOKENIZER)) before training."
    )
print("Tokenizer matches the original on real data -- the load-time regex warning is harmless here.")

### C5. Truncation report

Pairs longer than the budget get silently truncated by the trainer. This makes that visible. If more than ~5% are affected, raise `MAX_LENGTH` / `MAX_PROMPT_LENGTH` (memory permitting).

In [ ]:
def n_tokens(text):
    return len(tokenizer(text, add_special_tokens=False).input_ids)


prompt_lens, total_lens = [], []
for ex in train_ds:
    p = n_tokens(ex["prompt"])
    c = max(n_tokens(ex["chosen"]), n_tokens(ex["rejected"])) + 1
    prompt_lens.append(p)
    total_lens.append(p + c)

over_prompt = sum(p > MAX_PROMPT_LENGTH for p in prompt_lens)
over_total = sum(t > MAX_LENGTH for t in total_lens)
print(f"Prompt tokens  p50={int(np.percentile(prompt_lens, 50))}  p95={int(np.percentile(prompt_lens, 95))}  max={max(prompt_lens)}")
print(f"Total tokens   p50={int(np.percentile(total_lens, 50))}  p95={int(np.percentile(total_lens, 95))}  max={max(total_lens)}")
print(f"Prompts over MAX_PROMPT_LENGTH ({MAX_PROMPT_LENGTH}): {over_prompt} ({over_prompt / len(train_ds):.1%})")
print(f"Pairs over MAX_LENGTH ({MAX_LENGTH}): {over_total} ({over_total / len(train_ds):.1%})")
if over_total / len(train_ds) > 0.05:
    print("WARNING: more than 5% of pairs will be truncated -- consider raising the length budget.")

---
## Part D — Baseline (the plain SFT model, measured before any training)

### D1. Evaluation helpers

**Held-out preference accuracy.** For each held-out pair: does the model give Ahmed's real reply a higher average per-token log-probability than the wrong reply? This measures the model directly, so unlike DPO's internal reward accuracy, it has a meaningful value before training starts. Average per-token log-probability (not the total) is the primary number, so reply length doesn't decide the winner.

**Generation.** Decoding is greedy (`do_sample=False`), so before/after differences come from the weights, not sampling randomness. Only the new tokens are decoded (slicing by token, not by string length).

In [ ]:
from tqdm.auto import tqdm


@torch.no_grad()
def completion_logprob(prompt, completion):
    p_ids = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").input_ids
    c_ids = tokenizer(completion, add_special_tokens=False, return_tensors="pt").input_ids
    ids = torch.cat([p_ids, c_ids], dim=1)[:, -MAX_LENGTH:].to("cuda")
    n_completion = min(c_ids.shape[1], ids.shape[1] - 1)
    logits = model(input_ids=ids).logits[0, :-1].float()
    token_lp = torch.log_softmax(logits, dim=-1).gather(1, ids[0, 1:, None]).squeeze(1)
    completion_lp = token_lp[-n_completion:]
    return completion_lp.mean().item(), completion_lp.sum().item()


def preference_accuracy(pairs, n):
    FastLanguageModel.for_inference(model)
    rows = pairs.select(range(min(n, len(pairs))))
    wins, wins_sum, margins = [], 0, []
    for ex in tqdm(rows, desc="Scoring held-out pairs"):
        c_mean, c_sum = completion_logprob(ex["prompt"], ex["chosen"])
        r_mean, r_sum = completion_logprob(ex["prompt"], ex["rejected"])
        wins.append(bool(c_mean > r_mean))
        wins_sum += c_sum > r_sum
        margins.append(c_mean - r_mean)
    k = len(rows)
    return {
        "accuracy_mean_logp": sum(wins) / k,
        "accuracy_sum_logp": wins_sum / k,
        "avg_margin_mean_logp": float(np.mean(margins)),
        "n_pairs": k,
        "per_pair_wins": wins,   # kept so before/after can be compared pair by pair
    }


@torch.no_grad()
def generate_reply(prompt_text):
    FastLanguageModel.for_inference(model)
    enc = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(
        **enc,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.1,
        no_repeat_ngram_size=4,
        eos_token_id=EOS_ID,
        pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else EOS_ID,
    )
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def handwritten_prompt(user_text):
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_MESSAGE}, {"role": "user", "content": user_text}],
        tokenize=False, add_generation_prompt=True,
    )


def paired_test(wins_a, wins_b):
    # Exact McNemar test on the same pairs scored twice (a = before/other, b = after/this).
    # Returns (fixed, broken, p): pairs that went wrong->right, right->wrong, and the
    # two-sided p-value under "no real difference".
    from math import comb
    assert len(wins_a) == len(wins_b), "Paired test needs the same pairs on both sides."
    fixed = sum((not a) and b for a, b in zip(wins_a, wins_b))
    broken = sum(a and (not b) for a, b in zip(wins_a, wins_b))
    n = fixed + broken
    p = min(1.0, 2 * sum(comb(n, i) for i in range(min(fixed, broken) + 1)) / 2 ** n) if n else 1.0
    return fixed, broken, p


# The test set is the only thing used to report results. Frozen here so nothing later
# (e.g. E2 rebuilding datasets) can change what "before" and "after" are measured on.
SCORING_PAIRS = test_ds
heldout_examples = test_ds.select(range(min(N_HELDOUT_GENERATIONS, len(test_ds))))

### D2. Measure the baseline

Takes a few minutes on a T4. Everything after training is compared against these numbers.

In [ ]:
baseline = {"preference": preference_accuracy(SCORING_PAIRS, len(SCORING_PAIRS))}
print("Baseline held-out preference:", json.dumps({k: v for k, v in baseline["preference"].items() if k != "per_pair_wins"}, indent=2))

baseline["handwritten"] = [generate_reply(handwritten_prompt(m)) for m in HANDWRITTEN_TEST_MESSAGES]
baseline["heldout"] = [generate_reply(ex["prompt"]) for ex in heldout_examples]

for msg, out in zip(HANDWRITTEN_TEST_MESSAGES, baseline["handwritten"]):
    print(f"\nUSER: {msg}\nSFT:  {out}")

---
## Part E — Train

### E1. Build the trainer

- **`beta=0.2`, `rpo_alpha=0.5`.** A lower beta lets the model move further from SFT. The `rpo_alpha` term adds an SFT loss on `chosen`, so that movement can't come from making Ahmed's real replies less likely (the v6 failure). If this TRL version doesn't support `rpo_alpha`, the cell says so loudly and continues as plain DPO.
- **Best-model tracking.** Evaluation and checkpointing happen on the same schedule. The checkpoint with the lowest eval loss is restored at the end. Early stopping ends the run once eval loss stops improving.
- **`ref_model=None`.** With LoRA, the base model with adapters disabled serves as the frozen reference, so no second copy is needed.

**About `tokenizer=` vs `processing_class=`.** Newer TRL renamed this argument; the cell tries both.

In [ ]:
from trl import DPOConfig, DPOTrainer
from transformers import EarlyStoppingCallback


def make_dpo_config(include_rpo):
    kwargs = dict(
        output_dir=OUTPUT_DIR,
        beta=HP["beta"],
        max_length=MAX_LENGTH,
        max_prompt_length=MAX_PROMPT_LENGTH,
        learning_rate=HP["learning_rate"],
        lr_scheduler_type="linear",
        num_train_epochs=HP["num_train_epochs"],
        per_device_train_batch_size=HP["per_device_train_batch_size"],
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=HP["gradient_accumulation_steps"],
        warmup_steps=HP["warmup_steps"],
        optim="adamw_8bit",
        weight_decay=0.01,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        eval_strategy="steps",
        eval_steps=HP["eval_steps"],
        save_strategy="steps",
        save_steps=HP["eval_steps"],
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        seed=SEED,
    )
    if include_rpo and HP["rpo_alpha"]:
        kwargs["rpo_alpha"] = HP["rpo_alpha"]
    return DPOConfig(**kwargs)


try:
    dpo_args = make_dpo_config(include_rpo=True)
    RPO_ACTIVE = bool(HP["rpo_alpha"])
except TypeError as err:
    print("!" * 70)
    print("WARNING: this TRL version does not accept rpo_alpha -- running PLAIN DPO.")
    print("Likelihood displacement (logps/chosen falling) is NOT guarded against in this run.")
    print("Error:", err)
    print("!" * 70)
    dpo_args = make_dpo_config(include_rpo=False)
    RPO_ACTIVE = False

print("RPO anchor active:", RPO_ACTIVE)


def make_trainer(train_data, eval_data):
    common = dict(
        model=model, ref_model=None, args=dpo_args,
        train_dataset=train_data, eval_dataset=eval_data,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=HP["early_stopping_patience"])],
    )
    try:
        return DPOTrainer(tokenizer=tokenizer, **common)
    except TypeError:
        return DPOTrainer(processing_class=tokenizer, **common)

### E2. Verify EOS handling on the actual tokenized data

The reply must end with exactly one end-of-turn token. With none, the model never learns where to stop. With two, it learns a strange double ending. Some TRL versions append EOS automatically and some don't, so this cell inspects the tokenized data instead of assuming. It adds EOS only if it's missing, and stops on a double EOS.

In [ ]:
def eos_status(trainer):
    ds = trainer.train_dataset
    if "chosen_input_ids" not in ds.column_names:
        return "unknown"
    ids = ds[0]["chosen_input_ids"]
    eos = tokenizer.eos_token_id
    if len(ids) >= 2 and ids[-1] == eos and ids[-2] == eos:
        return "double"
    return "ok" if ids[-1] == eos else "missing"


def add_eos(ex):
    ex["chosen"] = ex["chosen"] + tokenizer.eos_token
    ex["rejected"] = ex["rejected"] + tokenizer.eos_token
    return ex


trainer = make_trainer(train_ds, dev_ds)
status = eos_status(trainer)
print("EOS status with raw data:", status)

if status == "missing":
    print("Trainer does not append EOS -- adding it to chosen/rejected and rebuilding.")
    train_ds, dev_ds = train_ds.map(add_eos), dev_ds.map(add_eos)
    trainer = make_trainer(train_ds, dev_ds)
    status = eos_status(trainer)
    print("EOS status after fix:", status)

assert status != "double", "Double EOS detected -- remove any manual EOS appending from the dataset."
assert status != "missing", "EOS still missing after the fix -- inspect trainer.train_dataset[0]."
if status == "unknown":
    print("Could not inspect tokenized columns in this TRL version; EOS left to the trainer's default.")

### E3. Train (resumes automatically after a disconnect)

First, one evaluation at step 0 (the plain SFT model) records the true starting point for the diagnosis. Then training runs.

If Colab disconnects, re-run every cell from the top with the **same `RUN_NAME`**. This cell picks up from the latest checkpoint in this run's folder instead of starting over.

In [ ]:
assert not (RUN_ALREADY_FINISHED and not OVERWRITE_EXISTING_RUN), (
    f"Run '{RUN_NAME}' already finished -- refusing to overwrite it. Set a new RUN_NAME, "
    "or OVERWRITE_EXISTING_RUN = True on purpose."
)
# Colab's progress-bar callback (and early stopping) refuse to run before training starts,
# so they're detached for this single evaluation and re-attached right after.
_step0_blockers = [cb for cb in trainer.callback_handler.callbacks
                   if type(cb).__name__ in ("NotebookProgressCallback", "EarlyStoppingCallback")]
for cb in _step0_blockers:
    trainer.remove_callback(cb)
try:
    initial_eval = trainer.evaluate()
    print(f"Step-0 dev eval: loss={initial_eval.get('eval_loss'):.4f}  "
          f"logps/chosen={initial_eval.get('eval_logps/chosen', float('nan')):.2f}")
except Exception as err:   # never block training on this diagnostic
    initial_eval = None
    print("Step-0 evaluation failed (diagnosis will start from the first in-training eval):", err)
finally:
    for cb in _step0_blockers:
        trainer.add_callback(cb)

FastLanguageModel.for_training(model)

existing = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
resume_from = existing[-1] if existing else None
print("Resuming from:", resume_from or "nothing -- fresh start")

train_result = trainer.train(resume_from_checkpoint=resume_from)

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best eval loss:", trainer.state.best_metric)
print("Stopped at step:", trainer.state.global_step, "of max", trainer.state.max_steps)

# Prove the best checkpoint is what's actually loaded -- not just what the trainer reports.
best_ckpt = trainer.state.best_model_checkpoint
best_file = os.path.join(best_ckpt, "adapter_model.safetensors") if best_ckpt else None
if best_file and os.path.exists(best_file):
    expected, _ = adapter_file_fp(best_file)
    current = lora_fp(model)
    if abs(current - expected) / expected < 1e-3:
        print(f"Verified: model weights match {os.path.basename(best_ckpt)} on disk.")
    else:
        print(f"Restore did NOT take effect (model {current:.2f} vs file {expected:.2f}) -- loading it explicitly.")
        load_adapter_weights(best_file)
        print(f"Fixed: model now holds {os.path.basename(best_ckpt)}.")
else:
    print("WARNING: could not find the best checkpoint's adapter file to verify against:", best_ckpt)

In [ ]:
fp_after = lora_fp(model)
diff = abs(fp_after - fp_before)
print(f"LoRA fingerprint BEFORE: {fp_before:.2f}")
print(f"LoRA fingerprint AFTER:  {fp_after:.2f}  (best checkpoint restored)")
print(f"Diff: {diff:.2f} ({diff / fp_before:.3%})")
assert diff > 50, "LoRA weights barely changed -- training was effectively a no-op. Check learning rate / data."

---
## Part F — Evaluate and diagnose

### F1. After-training measurements (same held-out pairs, same prompts, same decoding)

In [ ]:
after = {"preference": preference_accuracy(SCORING_PAIRS, len(SCORING_PAIRS))}
assert len(after["preference"]["per_pair_wins"]) == len(baseline["preference"]["per_pair_wins"]), "Before/after scored different pairs."
after["handwritten"] = [generate_reply(handwritten_prompt(m)) for m in HANDWRITTEN_TEST_MESSAGES]
after["heldout"] = [generate_reply(ex["prompt"]) for ex in heldout_examples]

b, a = baseline["preference"], after["preference"]
print("Held-out preference accuracy (mean log-prob):")
print(f"  SFT baseline: {b['accuracy_mean_logp']:.1%}   ->   DPO: {a['accuracy_mean_logp']:.1%}   "
      f"(change {a['accuracy_mean_logp'] - b['accuracy_mean_logp']:+.1%})")
print(f"  Avg margin:   {b['avg_margin_mean_logp']:+.3f}   ->   {a['avg_margin_mean_logp']:+.3f}")

In [ ]:
print("=" * 70)
print("HELD-OUT CONVERSATIONS (from validation data -- the model never saw these)")
print("=" * 70)
for ex, before_out, after_out in zip(heldout_examples, baseline["heldout"], after["heldout"]):
    last_turn = ex["prompt"].rsplit("<|im_start|>user\n", 1)[-1].split("<|im_end|>")[0]
    print(f"\nLAST MESSAGE: {last_turn[:150]}")
    print(f"REAL AHMED:   {ex['chosen'].replace(tokenizer.eos_token, '')}")
    print(f"SFT:          {before_out}")
    print(f"DPO:          {after_out}")

print("\n" + "=" * 70)
print("HANDWRITTEN PROMPTS")
print("=" * 70)
for msg, before_out, after_out in zip(HANDWRITTEN_TEST_MESSAGES, baseline["handwritten"], after["handwritten"]):
    print(f"\nUSER: {msg}\nSFT:  {before_out}\nDPO:  {after_out}")

### F2. Automatic diagnosis

This reads the training history and the before/after numbers, then reports what happened and which knob to turn (see the decision guide in A2). The goal is that the next experiment is chosen by evidence, not guesswork.

In [ ]:
history = trainer.state.log_history
evals = [h for h in history if "eval_loss" in h]
findings = []

def note(level, title, detail):
    findings.append({"level": level, "title": title, "detail": detail})

if evals:
    max_step = max(h["step"] for h in evals)
    early_acc = [h.get("eval_rewards/accuracies") for h in evals if h["step"] <= 0.25 * max_step]
    early_acc = [x for x in early_acc if x is not None]
    if early_acc and max(early_acc) >= 0.98:
        note("WARN", "Dataset may be too easy",
             f"Eval reward accuracy hit {max(early_acc):.0%} in the first quarter of training. "
             "Tighten the builder (--max-similarity 0.4) for harder negatives.")

    start = initial_eval if initial_eval and "eval_logps/chosen" in initial_eval else evals[0]
    first_chosen = start.get("eval_logps/chosen")
    best = min(evals, key=lambda h: h["eval_loss"])
    best_chosen = best.get("eval_logps/chosen")
    if first_chosen is not None and best_chosen is not None:
        rel = (best_chosen - first_chosen) / abs(first_chosen)
        if rel < -0.05:
            note("WARN", "Real replies are becoming less likely (likelihood displacement)",
                 f"eval logps/chosen moved {first_chosen:.1f} -> {best_chosen:.1f} ({rel:+.1%}). "
                 + ("Raise rpo_alpha (e.g. 0.5 -> 1.0)." if RPO_ACTIVE else "rpo_alpha was not active -- enable it."))
        else:
            note("OK", "Real replies stayed likely",
                 f"eval logps/chosen {first_chosen:.1f} -> {best_chosen:.1f} ({rel:+.1%}).")

    if best["step"] < max_step:
        note("INFO", "Best checkpoint was before the end",
             f"Best eval loss at step {best['step']} of {max_step}; it was restored automatically.")
    if initial_eval and best["eval_loss"] >= initial_eval["eval_loss"]:
        note("WARN", "Training never beat the starting point",
             f"Best dev loss {best['eval_loss']:.4f} vs step-0 {initial_eval['eval_loss']:.4f}. Lower learning_rate.")
    elif best["step"] == evals[0]["step"]:
        note("WARN", "Best checkpoint was the first evaluation",
             "The model got worse from the first eval on. Lower learning_rate.")

# Paired comparison: the same held-out pairs were scored before and after, so count the pairs
# that flipped. Exact McNemar test: under "no real change", flips in either direction are equally likely.
before_wins = baseline["preference"]["per_pair_wins"]
after_wins = after["preference"]["per_pair_wins"]
fixed, broken, p_value = paired_test(before_wins, after_wins)
delta = after["preference"]["accuracy_mean_logp"] - baseline["preference"]["accuracy_mean_logp"]
flip_text = f"{delta:+.1%} ({fixed} pairs fixed, {broken} broken, p={p_value:.3f}, n={len(before_wins)})"

if delta < 0 and p_value < 0.05:
    note("WARN", "Held-out preference accuracy went DOWN (significant)",
         f"{flip_text}. Likely overfitting or data mismatch: lower learning_rate, check the overlap report in C2.")
elif p_value < 0.05:
    note("OK", "DPO improved held-out preference accuracy (significant)", f"{flip_text}.")
elif delta > 0:
    note("INFO", "Improvement is within noise",
         f"{flip_text}. Promising but not proven. Lower beta (currently {HP['beta']}), or build a bigger eval set "
         "(more pairs from validation.jsonl) to get a clear answer.")
else:
    note("WARN", "No measurable effect on held-out data",
         f"{flip_text}. Lower beta (currently {HP['beta']}) or build more training pairs.")

unchanged = sum(x == y for x, y in zip(baseline["handwritten"] + baseline["heldout"], after["handwritten"] + after["heldout"]))
total = len(baseline["handwritten"]) + len(baseline["heldout"])
if unchanged > total / 2:
    note("INFO", "Most generations are unchanged",
         f"{unchanged}/{total} identical under greedy decoding. Expected if the held-out delta is small; "
         "lowering beta is the main lever.")

if not CLEAN_EVAL:
    note("WARN", "Eval set is not clean",
         "Dev/test pairs came from the SFT training data. Build them from validation.jsonl.")
elif not SEPARATE_TEST:
    note("WARN", "No separate test set",
         "Test pairs were split off the dev set. Build dpo_test_hard.jsonl for a clean, larger test set.")

for f in findings:
    print(f"[{f['level']}] {f['title']}\n       {f['detail']}\n")

---
## Part G — Save

### G1. Save the adapter (the best checkpoint, already restored) and verify it

Only the LoRA adapter (~250 MB) is saved, not a full 6–7 GB merge. Load it later with `PeftModel.from_pretrained(base_model, ADAPTER_DIR)`. The size check catches the earlier failure mode where only config files were written.

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

files = os.listdir(ADAPTER_DIR)
total_mb = sum(os.path.getsize(os.path.join(ADAPTER_DIR, f)) for f in files if os.path.isfile(os.path.join(ADAPTER_DIR, f))) / 1e6
weights = [f for f in files if f.startswith("adapter_model") and f.endswith((".safetensors", ".bin"))]
print("Files:", files)
print(f"Total size: {total_mb:.1f} MB | weight files: {weights}")
assert weights and total_mb > 50, "Adapter save looks incomplete -- do NOT delete checkpoints."
print("Verified: adapter saved to", ADAPTER_DIR)

### G2. Write the run summary

Saves the config, metrics, best checkpoint, diagnosis, and sample generations to `run_summary.json` in the run folder. Comparing experiments becomes a matter of comparing two JSON files instead of digging through old notebook outputs.

In [ ]:
summary = {
    "run_name": RUN_NAME,
    "base_checkpoint": BASE_CHECKPOINT_PATH,
    "train_file": TRAIN_JSONL_PATH,
    "dev_file": DEV_JSONL_PATH if CLEAN_EVAL else "split from train (NOT clean)",
    "test_file": TEST_JSONL_PATH if SEPARATE_TEST else "split from dev",
    "n_train_pairs": len(train_ds),
    "n_dev_pairs": len(dev_ds),
    "n_test_pairs": len(SCORING_PAIRS),
    "initial_dev_eval": initial_eval,
    "hyperparameters": HP,
    "rpo_active": RPO_ACTIVE,
    "max_length": MAX_LENGTH,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_eval_loss": trainer.state.best_metric,
    "stopped_at_step": trainer.state.global_step,
    "lora_fingerprint": {"before": fp_before, "after": fp_after},
    "heldout_preference": {"sft_baseline": baseline["preference"], "dpo": after["preference"]},
    "diagnosis": findings,
    "generations": {
        "handwritten": [{"user": m, "sft": s, "dpo": d} for m, s, d in zip(HANDWRITTEN_TEST_MESSAGES, baseline["handwritten"], after["handwritten"])],
        "heldout": [{"real": ex["chosen"], "sft": s, "dpo": d} for ex, s, d in zip(heldout_examples, baseline["heldout"], after["heldout"])],
    },
    "log_history": history,
}
with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("Saved run summary to", SUMMARY_PATH)

### G3. Head-to-head: this run vs. earlier runs, on the same test pairs

Each run's "change" is measured against its own baseline, which isn't enough to choose between runs. This cell loads each finished run in `COMPARE_WITH_RUNS` (e.g. v7) into the model, scores it on **this run's** test pairs, and runs the same paired test, so the comparison is like for like. Loads are verified by fingerprint, and this run's own weights are restored at the end.

**Choosing the winner:**
- One run is **significantly** better (p < 0.05) → that run wins.
- No significant difference → pick the more conservative run (higher `beta`, i.e. closer to the SFT model). Equal measured quality with less drift is the safer choice.
- If no run significantly beats SFT, report that honestly: DPO's effect on this data is small. The chosen adapter is still a valid, verified final model.

In [ ]:
head_to_head = []
this_wins = after["preference"]["per_pair_wins"]
this_acc = after["preference"]["accuracy_mean_logp"]

for other in COMPARE_WITH_RUNS:
    other_adapter = f"{DRIVE_ROOT}/DPO/runs/{other}/adapter"
    if other == RUN_NAME or not os.path.exists(os.path.join(other_adapter, "adapter_model.safetensors")):
        print(f"Skipping {other}: no saved adapter (or it is this run).")
        continue
    load_adapter_weights(other_adapter)
    other_pref = preference_accuracy(SCORING_PAIRS, len(SCORING_PAIRS))
    fixed_h, broken_h, p_h = paired_test(other_pref["per_pair_wins"], this_wins)
    diff = this_acc - other_pref["accuracy_mean_logp"]
    if p_h < 0.05:
        verdict = f"{RUN_NAME} is significantly {'better' if diff > 0 else 'worse'}"
    else:
        verdict = "no significant difference"
    head_to_head.append({"other_run": other, "other_accuracy": other_pref["accuracy_mean_logp"],
                         "this_accuracy": this_acc, "difference": diff,
                         "pairs_this_wins": fixed_h, "pairs_other_wins": broken_h, "p_value": p_h,
                         "verdict": verdict})
    print(f"{RUN_NAME} {this_acc:.1%} vs {other} {other_pref['accuracy_mean_logp']:.1%} "
          f"({diff:+.1%}; {fixed_h} pairs favor {RUN_NAME}, {broken_h} favor {other}, p={p_h:.3f}) -> {verdict}")

# Put this run's own (verified) weights back.
load_adapter_weights(ADAPTER_DIR)
print("Restored this run's adapter:", ADAPTER_DIR)

summary["head_to_head"] = head_to_head
with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("Head-to-head results added to", SUMMARY_PATH)

### G4. Optional — delete checkpoints (manual, after you've reviewed the results)

Not automatic anymore. In an earlier run, automatic cleanup deleted the checkpoint that turned out to be the best one. Flip the flag only once you're satisfied with the run.

In [ ]:
DELETE_CHECKPOINTS = False

if DELETE_CHECKPOINTS:
    assert os.path.exists(os.path.join(ADAPTER_DIR, "adapter_model.safetensors")), "Adapter not saved -- refusing to delete."
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    print("Deleted", OUTPUT_DIR)
else:
    print("Checkpoints kept at", OUTPUT_DIR)

### G5. Optional — merge into a standalone model (needs ~7 GB free)

Only merge the run you've decided is the best one. This cell frees the training model from GPU memory first, then loads the adapter folder (Unsloth reads the base model path from the adapter's config) and merges. **To merge a different run (e.g. v7) in a fresh session:** run the Drive-mount cell, the install cell, A1, A2 (with `RUN_NAME` set to the winning run, e.g. `"dpo_v7_rpo"`), and A3, then this cell. You can skip Parts B–G4. If the merge runs out of GPU memory right after training, use this fresh-session route: it starts with an empty GPU.

In [ ]:
free_gb = shutil.disk_usage(DRIVE_ROOT).free / 1e9
print(f"Free space: {free_gb:.2f} GB")
assert free_gb > 7, f"Only {free_gb:.2f} GB free -- a merged fp16 3B model needs ~6-7 GB."

import gc

# Free the training model first: two fp16 3B models don't fit on a T4 together.
for name in ("trainer", "model", "merge_model", "merge_base"):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

# Unsloth loads base model + adapter in one call when given the adapter folder
# (the adapter's config records the base model path). Loading via
# peft.PeftModel.from_pretrained makes save_pretrained_merged fail.
merge_model, merge_tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=False,
)
merge_model.save_pretrained_merged(MERGED_MODEL_DIR, merge_tokenizer, save_method="merged_16bit")

weights = [f for f in os.listdir(MERGED_MODEL_DIR) if f.endswith(".safetensors") and "index" not in f]
size_gb = sum(os.path.getsize(os.path.join(MERGED_MODEL_DIR, f)) for f in os.listdir(MERGED_MODEL_DIR)) / 1e9
assert weights and size_gb > 3, f"Merged model looks incomplete ({size_gb:.2f} GB)."
print(f"Saved and verified merged model ({size_gb:.2f} GB) at", MERGED_MODEL_DIR)

### G6. Compare all runs

Reads every `run_summary.json` in `DPO/runs/` and prints one table, so choosing the best run is a single glance. Works in a fresh session too: only Part A is needed first.

**Caveat:** runs scored on different pairs (v7: 150 dev pairs; v8: the separate test set) aren't directly comparable in this table. Use it as an overview; **G3's head-to-head is the fair comparison** for choosing a winner.

In [ ]:
import glob, json

rows = []
for path in sorted(glob.glob(f"{DRIVE_ROOT}/DPO/runs/*/run_summary.json")):
    s = json.load(open(path, encoding="utf-8"))
    pref = s["heldout_preference"]
    sft, dpo = pref["sft_baseline"], pref["dpo"]
    verdict = next((d["title"] for d in s["diagnosis"] if "held-out" in d["title"].lower() or "noise" in d["title"].lower() or "effect" in d["title"].lower()), "")
    rows.append((
        s["run_name"], s["hyperparameters"]["beta"], s["hyperparameters"].get("rpo_alpha"),
        f"{sft['accuracy_mean_logp']:.1%}", f"{dpo['accuracy_mean_logp']:.1%}",
        f"{dpo['accuracy_mean_logp'] - sft['accuracy_mean_logp']:+.1%}", dpo["n_pairs"],
        s["best_checkpoint"].rsplit("-", 1)[-1] if s.get("best_checkpoint") else "-", verdict,
    ))

header = ("run", "beta", "rpo", "SFT acc", "DPO acc", "change", "n", "best step", "verdict")
widths = [max(len(str(r[i])) for r in rows + [header]) for i in range(len(header))]
for r in [header] + rows:
    print("  ".join(str(v).ljust(w) for v, w in zip(r, widths)))

---
## Known limitations

- **Privacy.** Generated replies can contain real names from the source conversations (v6 outputs addressed the other person by name). The model and its outputs must stay private, the same as the data.
- **Eval size.** About 300 test pairs still leaves a few points of noise. The paired tests in F2 and G3 say whether a difference is real.
- **Dev/test come from the same conversations.** Both are drawn from `validation.jsonl`; different sliding windows of one conversation can land in both. Exact-prompt overlap is removed, and since dev only picks the checkpoint (via loss), the remaining leakage is minor.
- **Untested on newer library versions.** The `tokenizer`/`processing_class` and `rpo_alpha` fallbacks cover known TRL API changes. A future TRL release may need small adjustments, and those cells fail fast with a clear message rather than silently.